# 🔬 CellOneFormer: Evaluation, Hyperparameter Tuning & Test Benchmark
### Pipeline d'évaluation et d'inférence basé sur le *Multi-modality Cell Segmentation Challenge (Nature Methods 2024)*

Ce notebook permet de :
1. **Charger vos poids entraînés** (`oneformer_real_cells_best.pt` ou `oneformer_cells_best.pt`) directement depuis Google Drive.
2. **Optimiser les hyperparamètres de post-traitement (*Tuning*)** sur le jeu `tuning/` : seuil de confiance de détection (`score_threshold`), seuil de fusion des coutures (*Soft-IoU stitching*), et taille minimale d'aire cellulaire (`min_cell_area`).
3. **Calculer les métriques officielles du challenge** : $\text{AP}_{50}$, $\text{AP}_{75}$, $\text{mAP}_{[50:95]}$ et score F1.
4. **Générer les prédictions finales (*Testing*)** sur les images pleine résolution de `testing/` directement en mémoire GPU (sans recharger le modèle à chaque image).
5. **Visualiser les galeries de segmentation d'instances** (comparaison côte-à-côte haute résolution).
6. **Exporter les masques de soumission** automatiquement vers votre Google Drive.

## 1. Vérification du GPU & Montage de Google Drive
Assurez-vous d'être connecté à un GPU (T4 ou A100) : **Exécution > Modifier le type d'exécution > GPU**.

In [ ]:
!nvidia-smi

from google.colab import drive
import os

drive.mount('/content/drive')
print("✅ Google Drive connecté avec succès.")

## 2. Configuration de l'environnement & Dépendances
Clonage ou mise à jour du dépôt et installation des bibliothèques compatibles avec Python 3.13 / NumPy 2.x de Colab.

In [ ]:
import os
import sys

REPO_DIR = "/content/CellOneFormer"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/BaptFontaine/CellOneFormer.git "{REPO_DIR}"
else:
    !cd "{REPO_DIR}" && git pull

%cd /content/CellOneFormer
%env PYTHONPATH=/content/CellOneFormer

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Optimisation de la mémoire GPU
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Conserve le NumPy 2.x préinstallé sur Colab (évite la recompilation de NumPy 2.0.2 sous Python 3.13)
!pip install -q "numpy>=2.0.0,<2.3.0" transformers accelerate albumentations tifffile imagecodecs
print("✅ Environnement CellOneFormer prêt !")

## 3. Chargement du Modèle & Vérification des Poids Entraînés
Récupération automatique du checkpoint (`oneformer_real_cells_best.pt` ou `oneformer_cells_best.pt`) sauvegardé sur votre Google Drive.

In [ ]:
import os
import shutil

DRIVE_CANDIDATES = [
    "/content/drive/MyDrive/cell_oneformer_real_checkpoints/oneformer_real_cells_best.pt",
    "/content/drive/MyDrive/cell_oneformer_checkpoints/oneformer_cells_best.pt",
]
LOCAL_CKPT_DIR = "/content/CellOneFormer/checkpoints"
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

DRIVE_CHECKPOINT = next((p for p in DRIVE_CANDIDATES if os.path.exists(p)), None)
LOCAL_CHECKPOINT = None

if DRIVE_CHECKPOINT is not None:
    LOCAL_CHECKPOINT = os.path.join(LOCAL_CKPT_DIR, os.path.basename(DRIVE_CHECKPOINT))
    if not os.path.exists(LOCAL_CHECKPOINT) or os.path.getsize(LOCAL_CHECKPOINT) != os.path.getsize(DRIVE_CHECKPOINT):
        shutil.copy2(DRIVE_CHECKPOINT, LOCAL_CHECKPOINT)
    size_mb = os.path.getsize(LOCAL_CHECKPOINT) / (1024 * 1024)
    print(f"✅ Checkpoint chargé avec succès : {LOCAL_CHECKPOINT} ({size_mb:.1f} Mo)")
else:
    local_fallbacks = [
        os.path.join(LOCAL_CKPT_DIR, "oneformer_real_cells_best.pt"),
        os.path.join(LOCAL_CKPT_DIR, "oneformer_cells_best.pt"),
    ]
    LOCAL_CHECKPOINT = next((p for p in local_fallbacks if os.path.exists(p)), None)
    if LOCAL_CHECKPOINT is not None:
        size_mb = os.path.getsize(LOCAL_CHECKPOINT) / (1024 * 1024)
        print(f"✅ Checkpoint local trouvé : {LOCAL_CHECKPOINT} ({size_mb:.1f} Mo)")
    else:
        LOCAL_CHECKPOINT = os.path.join(LOCAL_CKPT_DIR, "oneformer_real_cells_best.pt")
        print("⚠️ Attention : Aucun checkpoint trouvé dans Google Drive ni en local.")
        print("Chemins vérifiés :", DRIVE_CANDIDATES)

## 4. Préparation & Détection des Données (`tuning` et `testing`)
Copie des images de validation et de test depuis Google Drive vers le SSD local de Colab pour maximiser les performances d'inférence.

In [ ]:
import os
import glob
from pathlib import Path
from src.training.train import find_image_mask_pairs

DRIVE_DATASET = "/content/drive/MyDrive/DatasetCellSeg"
LOCAL_DATASET = "/content/CellOneFormer/data/raw/DatasetCellSeg"
os.makedirs(LOCAL_DATASET, exist_ok=True)

def find_case_subdir(root, name):
    for cand in [name, name.capitalize(), name.lower(), name.upper()]:
        p = os.path.join(root, cand)
        if os.path.exists(p):
            return p
    return None

if os.path.exists(DRIVE_DATASET):
    print(f"📁 Synchronisation depuis {DRIVE_DATASET} vers {LOCAL_DATASET}...")
    for split in ["Tuning", "tuning", "Testing", "testing"]:
        src_split = os.path.join(DRIVE_DATASET, split)
        dst_split = os.path.join(LOCAL_DATASET, split)
        if os.path.exists(src_split):
            os.makedirs(dst_split, exist_ok=True)
            !cp -ru "{src_split}"/* "{dst_split}/"

valid_exts = ('.png', '.tif', '.tiff', '.jpg', '.jpeg', '.bmp')

def collect_split_pairs(split_name):
    split_dir = find_case_subdir(LOCAL_DATASET, split_name)
    if split_dir is None:
        return [], {}, []
    pairs = find_image_mask_pairs(Path(split_dir))
    if pairs:
        imgs = [str(img_p) for img_p, _ in pairs]
        mask_map = {str(img_p): str(msk_p) for img_p, msk_p in pairs if msk_p is not None and msk_p.exists()}
        masks = list(mask_map.values())
        return imgs, mask_map, masks
    # Recherche récursive (ex: Testing/Public/*.*)
    all_files = sorted(glob.glob(f"{split_dir}/**/*.*", recursive=True))
    imgs = [f for f in all_files if f.lower().endswith(valid_exts) and not any(k in os.path.basename(f).lower() for k in ['mask', 'label', '_gt'])]
    return imgs, {}, []

tuning_imgs, tuning_mask_by_img, tuning_masks = collect_split_pairs("tuning")
test_imgs, test_mask_by_img, _ = collect_split_pairs("testing")

# Fallback si tuning est vide : synchroniser un échantillon depuis training/
if len(tuning_imgs) == 0:
    src_train = find_case_subdir(DRIVE_DATASET, "training") if os.path.exists(DRIVE_DATASET) else None
    if src_train:
        dst_train = os.path.join(LOCAL_DATASET, os.path.basename(src_train))
        os.makedirs(dst_train, exist_ok=True)
        !cp -ru "{src_train}"/* "{dst_train}/"
    train_imgs, train_mask_map, train_masks = collect_split_pairs("training")
    if len(train_imgs) > 0:
        print("ℹ️ Pas d'images séparées dans tuning/. Utilisation d'un échantillon annoté de training/ pour calibrer les seuils.")
        tuning_imgs = train_imgs[:5]
        tuning_mask_by_img = {k: train_mask_map[k] for k in tuning_imgs if k in train_mask_map}
        tuning_masks = list(tuning_mask_by_img.values())

print(f"🔬 Images disponibles pour le Tuning (validation) : {len(tuning_imgs)} ({len(tuning_mask_by_img)} avec masque GT)")
print(f"🚀 Images disponibles pour le Test final         : {len(test_imgs)}")

## 5. Hyperparameter Tuning sur le Split de Validation
Pré-calcul unique de l'inférence GPU puis optimisation rapide des paramètres de post-traitement pour maximiser $\text{mAP}_{[50:95]}$ :
- `score_threshold` : Seuil de probabilité minimale pour conserver une cellule détectée.
- `iou_threshold_stitch` : Seuil Soft-IoU de déduplication des cellules chevauchant la couture de deux tuiles.
- `min_cell_area` : Filtrage des détections parasites de moins de $N$ pixels.

In [ ]:
import inspect
import yaml
import numpy as np
import torch

from src.dataset.tiler import SlidingWindowTiler, load_microscopy_image, load_instance_mask
from src.inference.predict import load_trained_model, infer_tiles_batched
from src.inference.stitcher import TileStitcher
from src.utils.metrics import compute_challenge_benchmark_metrics

# Grille d'hyperparamètres à évaluer
SCORE_THRESHOLDS = [0.2, 0.25, 0.3, 0.4, 0.5, 0.6]
STITCH_IOUS = [0.4, 0.5, 0.6]
MIN_AREAS = [15, 30]
MAX_EVAL_SAMPLES = 10  # Nombre d'images de validation utilisées pour le tuning

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with open("configs/config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

print(f"Chargement du modèle sur {device}...")
model_module = load_trained_model(cfg, LOCAL_CHECKPOINT, device)
print(f"✅ Poids chargés depuis : {LOCAL_CHECKPOINT}")

tile_size = int(cfg["inference"].get("tile_size", 512))
tile_overlap = int(cfg["inference"].get("tile_overlap", 96))
batch_size = int(cfg["inference"].get("batch_size", 4))
max_cell_area = int(cfg["data"].get("max_cell_area", 8000))
percentile_norm = cfg["data"].get("percentile_norm", True)
apply_clahe = cfg["data"].get("apply_clahe", False)
gaussian_seam = bool(cfg["inference"].get("gaussian_seam_weighting", True))
task_name = cfg["model"].get("task", "instance").lower()
task_key = "instance" if "instance" in task_name else ("semantic" if "semantic" in task_name else "panoptic")

# 1. Pré-calcul unique des prédictions GPU par tuile
tiler = SlidingWindowTiler(tile_size=tile_size, tile_overlap=tile_overlap, min_cell_area=min(MIN_AREAS))
min_score_grid = min(SCORE_THRESHOLDS)
min_area_grid = min(MIN_AREAS)
cached_eval_samples = []

eval_candidates = [p for p in tuning_imgs if p in tuning_mask_by_img][:MAX_EVAL_SAMPLES]
print(f"⚡ Inférence GPU mise en cache sur {len(eval_candidates)} image(s) de validation...")

for img_p in eval_candidates:
    gt_mask = load_instance_mask(tuning_mask_by_img[img_p])
    gt_ids = [gid for gid in np.unique(gt_mask) if gid > 0]
    if not gt_ids:
        continue
    gt_instances = np.array([gt_mask == gid for gid in gt_ids], dtype=bool)

    image = load_microscopy_image(img_p, percentile_norm=percentile_norm, apply_clahe=apply_clahe)
    h, w = image.shape[:2]
    tiles = tiler.tile_image_and_mask(image)

    tile_preds = infer_tiles_batched(
        tiles=tiles,
        model_module=model_module,
        device=device,
        task_key=task_key,
        batch_size=batch_size,
        score_thresh=min_score_grid,
        min_area=min_area_grid,
        max_area=max_cell_area,
    )
    for tp in tile_preds:
        tp["areas"] = [int(np.sum(m)) for m in tp["masks"]]

    cached_eval_samples.append({"h": h, "w": w, "gt_instances": gt_instances, "tiles": tile_preds})

def evaluate_tuning_config(score_th, stitch_iou, min_area):
    all_metrics = []
    stitcher = TileStitcher(
        iou_threshold=stitch_iou,
        min_area=min_area,
        max_area=max_cell_area,
        gaussian_seam_weighting=gaussian_seam,
    )
    for sample in cached_eval_samples:
        filtered_tiles = []
        for t in sample["tiles"]:
            f_masks, f_scores = [], []
            for m, s, a in zip(t["masks"], t["scores"], t["areas"]):
                if s >= score_th and min_area <= a <= max_cell_area:
                    f_masks.append(m)
                    f_scores.append(s)
            filtered_tiles.append({"masks": f_masks, "scores": f_scores, "coords": t["coords"]})

        _, kept_masks = stitcher.stitch_tile_predictions(sample["h"], sample["w"], filtered_tiles)
        metrics = compute_challenge_benchmark_metrics(kept_masks, sample["gt_instances"])
        all_metrics.append(metrics)

    if not all_metrics:
        return {"AP@50": 0.0, "AP@75": 0.0, "mAP@[50:95]": 0.0, "F1": 0.0}

    return {
        "AP@50": float(np.mean([m.get("AP@50", 0.0) for m in all_metrics])),
        "AP@75": float(np.mean([m.get("AP@75", 0.0) for m in all_metrics])),
        "mAP@[50:95]": float(np.mean([m.get("mAP@[50:95]", 0.0) for m in all_metrics])),
        "F1": float(np.mean([m.get("F1", m.get("mean_f1", 0.0)) for m in all_metrics])),
    }

best_config = {
    "score_threshold": cfg["inference"].get("score_threshold", 0.25),
    "stitch_iou": cfg["inference"].get("iou_threshold_stitch", 0.5),
    "min_area": cfg["data"].get("min_cell_area", 15),
    "mAP@[50:95]": 0.0,
}

if cached_eval_samples:
    print(f"🔍 Évaluation de la grille sur {len(cached_eval_samples)} image(s) annotée(s)...")
    tuning_results = []
    for s_th in SCORE_THRESHOLDS:
        for st_iou in STITCH_IOUS:
            for m_area in MIN_AREAS:
                res = evaluate_tuning_config(score_th=s_th, stitch_iou=st_iou, min_area=m_area)
                tuning_results.append({
                    "score_threshold": s_th,
                    "stitch_iou": st_iou,
                    "min_area": m_area,
                    **res,
                })

    tuning_results.sort(key=lambda x: (x.get("mAP@[50:95]", 0.0), x.get("AP@50", 0.0)), reverse=True)
    print("\n🏆 Résultats du Tuning (classés par mAP@[50:95]) :\n")
    print("=" * 86)
    print(f" {'Score Th':<9} | {'Stitch IoU':<10} | {'Min Area':<8} | {'AP@50':<10} | {'AP@75':<10} | {'mAP@[50:95]':<12} | {'F1':<8}")
    print("-" * 86)
    for r in tuning_results[:15]:
        print(f" {r['score_threshold']:<9.2f} | {r['stitch_iou']:<10.2f} | {r['min_area']:<8d} | {r['AP@50']:<10.4f} | {r['AP@75']:<10.4f} | {r['mAP@[50:95]']:<12.4f} | {r['F1']:<8.4f}")
    print("=" * 86)
    best_config = tuning_results[0]
else:
    print("\nℹ️ Aucun masque de vérité terrain apparié : utilisation des hyperparamètres par défaut de config.yaml.")

print(f"\n⭐ Profil optimal : score_threshold={best_config['score_threshold']}, stitch_iou={best_config['stitch_iou']}, min_area={best_config['min_area']}")

## 6. Inférence & Benchmark sur le Jeu de Test (`testing/`)
Applique les paramètres optimaux directement en mémoire GPU (sans relancer Python pour chaque image) afin de traiter rapidement toutes les images de test.

In [ ]:
import os
import glob
import cv2
import yaml
import numpy as np
import torch
from tqdm.auto import tqdm

from src.dataset.tiler import load_microscopy_image, load_instance_mask
from src.inference.predict import load_trained_model, predict_full_image, create_overlay
from src.utils.metrics import compute_challenge_benchmark_metrics

OUTPUT_PRED_DIR = "/content/CellOneFormer/predictions_test"
os.makedirs(OUTPUT_PRED_DIR, exist_ok=True)

# Optionnel : limitez à un sous-ensemble (ex: 50) pour un aperçu rapide, ou laissez None pour tout traiter
MAX_TEST_IMAGES = None

if "cfg" not in globals():
    with open("configs/config.yaml", "r") as f:
        cfg = yaml.safe_load(f)
if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if "model_module" not in globals():
    model_module = load_trained_model(cfg, LOCAL_CHECKPOINT, device)

default_cfg = globals().get("best_config", {})
best_score_th = float(default_cfg.get("score_threshold", cfg["inference"].get("score_threshold", 0.25)))
best_stitch_iou = float(default_cfg.get("stitch_iou", cfg["inference"].get("iou_threshold_stitch", 0.5)))
best_min_area = int(default_cfg.get("min_area", cfg["data"].get("min_cell_area", 15)))

target_imgs = test_imgs if len(test_imgs) > 0 else tuning_imgs
if MAX_TEST_IMAGES is not None:
    target_imgs = target_imgs[:MAX_TEST_IMAGES]

percentile_norm = cfg["data"].get("percentile_norm", True)
apply_clahe = cfg["data"].get("apply_clahe", False)
eval_map = {**tuning_mask_by_img, **test_mask_by_img}
eval_metrics = []

print(f"🚀 Inférence en mémoire sur {len(target_imgs)} image(s) pleine résolution :")
print(f"   - Score Threshold : {best_score_th}")
print(f"   - Soft-IoU Stitch : {best_stitch_iou}")
print(f"   - Min Cell Area   : {best_min_area} px\n")

for img_path in tqdm(target_imgs, desc="Inférence CellOneFormer"):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    image = load_microscopy_image(img_path, percentile_norm=percentile_norm, apply_clahe=apply_clahe)

    label_matrix, kept_masks = predict_full_image(
        image=image,
        model_module=model_module,
        cfg=cfg,
        device=device,
        score_thresh=best_score_th,
        stitch_iou=best_stitch_iou,
        min_area=best_min_area,
    )

    out_mask_path = os.path.join(OUTPUT_PRED_DIR, f"{stem}_pred_mask.png")
    cv2.imwrite(out_mask_path, label_matrix.astype(np.uint16))

    overlay = create_overlay(image, label_matrix)
    out_vis_path = os.path.join(OUTPUT_PRED_DIR, f"{stem}_overlay.png")
    cv2.imwrite(out_vis_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    gt_path = eval_map.get(img_path)
    if gt_path and os.path.exists(gt_path):
        gt_mask = load_instance_mask(gt_path)
        gt_ids = [gid for gid in np.unique(gt_mask) if gid > 0]
        if gt_ids:
            gt_instances = np.array([gt_mask == gid for gid in gt_ids], dtype=bool)
            eval_metrics.append(compute_challenge_benchmark_metrics(kept_masks, gt_instances))

if eval_metrics:
    print("\n📊 Métriques moyennes sur les images avec vérité terrain :")
    print(f"   - AP@50      : {np.mean([m['AP@50'] for m in eval_metrics]):.4f}")
    print(f"   - AP@75      : {np.mean([m['AP@75'] for m in eval_metrics]):.4f}")
    print(f"   - mAP@[50:95]: {np.mean([m['mAP@[50:95]'] for m in eval_metrics]):.4f}")
    print(f"   - F1@0.5     : {np.mean([m['F1'] for m in eval_metrics]):.4f}")

print(f"\n✅ Inférence terminée ! {len(target_imgs)} image(s) enregistrée(s) dans {OUTPUT_PRED_DIR}.")

## 7. Galerie Visuelle des Résultats Haute Résolution
Affiche les images de microscopie originales et la segmentation d'instances reconstituée par CellOneFormer.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from src.dataset.tiler import load_microscopy_image

overlays = sorted(glob.glob(f"{OUTPUT_PRED_DIR}/*_overlay.png"))
target_by_stem = {os.path.splitext(os.path.basename(p))[0]: p for p in target_imgs}
num_to_display = min(4, len(overlays))

if num_to_display > 0:
    fig, axes = plt.subplots(num_to_display, 2, figsize=(16, 7 * num_to_display))
    if num_to_display == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(num_to_display):
        ov_p = overlays[i]
        stem = os.path.basename(ov_p).replace("_overlay.png", "")
        orig_path = target_by_stem.get(stem)

        if orig_path and os.path.exists(orig_path):
            orig_img = load_microscopy_image(orig_path, percentile_norm=percentile_norm, apply_clahe=apply_clahe)
            axes[i, 0].imshow(orig_img)
            axes[i, 0].set_title(f"Original : {stem}", fontsize=13)
            axes[i, 0].axis("off")

        ov_img = cv2.cvtColor(cv2.imread(ov_p), cv2.COLOR_BGR2RGB)
        axes[i, 1].imshow(ov_img)
        axes[i, 1].set_title(f"CellOneFormer Instances (Stitched) : {stem}", fontsize=13)
        axes[i, 1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Aucun overlay trouvé pour l'affichage.")

## 8. Exportation & Sauvegarde des Résultats sur Google Drive
Archive les masques prédits au format officiel (PNG 16-bit) et les prévisualisations directement sur votre Drive.

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DRIVE_EXPORT_DIR = f"/content/drive/MyDrive/CellOneFormer_Results_{timestamp}"
os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)

# Création d'une archive zip téléchargeable puis copie vers Google Drive
zip_name = f"/content/CellOneFormer_predictions_{timestamp}"
zip_path = shutil.make_archive(zip_name, 'zip', OUTPUT_PRED_DIR)
shutil.copy2(zip_path, DRIVE_EXPORT_DIR)

# Copie des fichiers individuels vers Google Drive
!cp -r "{OUTPUT_PRED_DIR}"/* "{DRIVE_EXPORT_DIR}/"

print("✅ Tous les résultats ont été exportés sur Google Drive dans :")
print(f"   📁 {DRIVE_EXPORT_DIR}")
print(f"   📦 Archive : {DRIVE_EXPORT_DIR}/CellOneFormer_predictions_{timestamp}.zip")